# Setup notebook

The steps in this section configure aspects of the notebook which includes:

1. Installing required libraries.
2. Enabling detailed logging.

## Install required libraries

In [ ]:
%pip install -r requirements.txt

## Setup detailed logging

This cell sets up DEBUG level logging which can help to troubleshoot issues. You can modify this from DEBUG to something lighter weight like ERROR if it's too much.

In [ ]:
# Setup logging in debug mode
import logging
import sys

def configure_logging(level="DEBUG"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """
    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

# 1. Creating Bot Service resource

When publishing a Foundry Agent to Microsoft Teams, the [Azure AI Bot Service](https://learn.microsoft.com/en-us/azure/bot-service/bot-service-overview?view=azure-bot-service-4.0) is used to proxy the communication between a Microsoft Teams client and Foundry Agent. This is done through a [channel](https://learn.microsoft.com/en-us/azure/bot-service/bot-builder-basics?view=azure-bot-service-4.0) for Microsoft Teams. 

The AI Bot Service resource created in Azure can be thought of as a metadata resource that establishes the relationship between the Foundry Agent and the Teams Channel that the AI Bot Service will proxy communication for. The Foundry Agent's Entra ID Agent Identity's principalId (or appId) is used to establish this relationship.

## Get an access token for the Microsoft Foundry API

First, the appId (principal_id) property of the Foundry Agent's Entra ID Agent ID needs to be captured from the Microsfot Foundry API. 

Before running this step you must log in to az cli. You will also need to retrieve your Entra ID tenant id. The Entra ID tenant id can be retrieved by running the command below after you log into az cli.

```az account show --query tenantId -o tsv```

In [ ]:
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Get a token for Foundry scope
credential = DefaultAzureCredential()
scopes = ["https://ai.azure.com"]

user_token = credential.get_token(*scopes)

## Get the principal id of the Foundry Agent's Entra ID Agent Identity

This step gets the principal_id (or appId) of the Entra ID Agent Identity associated with the Foundry Agent.

The user running this step must hold the Foundry User role over the Foundry account.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Function that gets the agent object
def get_foundry_agent(account_name: str, project_name: str, agent_name: str, token: str):
    """This function retrieves a Foundry agent by name from a Foundry project
    Args:
        account_name (str): The name of the Foundry account
        project_name (str): The name of the Foundry project
        agent_name (str): The name of the Foundry agent to retrieve
        token (str): The authentication token to use for the API request
    Returns:
        dict: The Foundry agent details if found, otherwise None
    """
    response = requests.get(
        f"https://{account_name}.services.ai.azure.com/api/projects/{project_name}/agents/{agent_name}?api-version=v1",
        headers={
            "Content-Type": "application/json", 
            "Authorization": f"Bearer {token}"
        }
    ) 

    if response.status_code == 200:
        return response.json()
    else:
        logging.error(f"Failed to retrieve agent: {response.status_code} - {response.text}")
        return None

# Grab the principal_id of the Entra ID Agent Identity associated with the Foundry Agent  
foundry_account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")
project_name = os.getenv("FOUNDRY_PROJECT_NAME")
agent_name = os.getenv("FOUNDRY_AGENT_NAME")

agent = get_foundry_agent(foundry_account_name, project_name, agent_name, user_token.token)
agent_principal_id = agent.get("instance_identity", {}).get("principal_id")
print(f"Foundry Agent Principal ID: {agent_principal_id}")

print(json.dumps(agent, indent=2))

## Create the Azure Bot Service Resource

You now need to create the Azure Bot Service resource that was discussed above. The resource will require the following inputs:

1. Your Entra ID tenant id
2. The principal_id of the Entra ID Agent Identity associated with the Foundry Agent.
3. The activity endpoint (more often referred to as messaging endpoint for other Bot Service integrations) of the Foundry Agent.


You should already have 1 and 2. 

For 3 the activity endpoint will look something like this:

**https://FOUNDRY_ACCOUNT_NAME.services.ai.azure.com/api/projects/PROJECT_NAME/agents/AGENT_NAME/endpoint/protocols/activityProtocol?api-version=2025-05-15-preview**

There is a [sample Terraform template](../../modules/bot-service/main.tf) in this repository.

To run this Terraform template the user must have the Contributor Role at either the Resource Group or Subscription.

# 2. Publishing the Agent

Once the Azure Bot Service resource has been created, the agent must now be configured to support the AI Bot Service integration. This is done through the enabling the activity endpoint for the agent and enabling the authorization scheme to support the AI Bot Service integration.



## Enable the activity endpoint and configure Bot Service authorization

The activity endpoint is the destination the Bot Service will direct messages from the Teams Service to. It must be activated. Additionally, the [authorization scheme](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/configure-agent?tabs=rest#authorization-schemes) of the endpoint needs to configured to support authorization for the Bot Service integration.

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Function that enables the activity protocol for the agent and configures the required Bot Service authorization scheme
def enable_agent_activity_protocol(account_name: str, project_name: str, agent_name: str, token: str):
    """This function enables the activity protocol for a Foundry agent and configures the required Bot Service authorization scheme
    Args:
        account_name (str): The name of the Foundry account
        project_name (str): The name of the Foundry project
        agent_name (str): The name of the Foundry agent to retrieve
        token (str): The authentication token to use for the API request
    Returns:
        dict: The updated Foundry agent details if the update was successful, otherwise None
    """

    # 
    body = {
        "agent_endpoint": {
            "protocols": [
                "responses",
                "activity"
            ],
            "authorization_schemes": [
                {
                    "type": "Entra",
                    "isolation_key_source": {
                        "kind": "Entra"
                    }
                },
                {
                    "type": "BotServiceRbac"
                }
            ]
        }
    }

    response = requests.patch(
        f"https://{account_name}.services.ai.azure.com/api/projects/{project_name}/agents/{agent_name}?api-version=v1",
        headers={
            "Content-Type": "application/merge-patch+json", 
            "Authorization": f"Bearer {token}",
            "Foundry-Features": "AgentEndpoints=V1Preview"
        },
        json=body
    ) 

    if response.status_code == 200:
        return response.json()
    else:
        logging.error(f"Failed to enable agent activity protocol: {response.status_code} - {response.text}")
        return None

# Grab the principal_id of the Entra ID Agent Identity associated with the Foundry Agent  
foundry_account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")
project_name = os.getenv("FOUNDRY_PROJECT_NAME")
agent_name = os.getenv("FOUNDRY_AGENT_NAME")

enabled_agent = enable_agent_activity_protocol(foundry_account_name, project_name, agent_name, user_token.token)
enabled_agent_guid = enabled_agent.get('versions', {}).get("latest", {}).get("agent_guid", {})
print(f"Enabled Agent GUID: {enabled_agent_guid}")
updated_agent_endpoint = enabled_agent.get('agent_endpoint', {})
print(f"Updated Agent Endpoint: {json.dumps(updated_agent_endpoint, indent=2)}")




## Publish the agent to Microsoft Teams

The final step is to publish the Foundry Agent to Microsoft Teams. When publishing the agent with the app_publish_scope set to **Individual** the agent will be published to the Microsoft 365 Agent Registry and immediately pushed as an  . If you publish the agent for all users in the Entra ID tenant with app_publish_scope set to **Tenant** the agent will be placed in a pending state in the [Microsoft 365 Agent Registry](https://learn.microsoft.com/en-us/microsoft-365/admin/manage/agent-registry?view=o365-worldwide).

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

def publish_agent_teams(
    subscription_id: str,
    resource_group: str,
    account_name: str, 
    project_name: str, 
    location: str,
    agent_name: str, 
    agent_guid: str,
    bot_id: str,
    app_publish_scope: str,
    publish_as_digital_worker: bool,
    app_version: str,
    short_description: str,
    full_description: str,
    developer_name: str,
    developer_website_url: str,
    privacy_url: str,
    terms_of_use_url: str,
    token: str
    ):
    """This function uses the Foundry API to publish a Foundry agent to Microsoft Teams
    Args:
        subscription_id (str): The Azure subscription ID where the Foundry account is provisioned
        resource_group (str): The name of the resource group where the Foundry account is provisioned
        account_name (str): The name of the Foundry account
        project_name (str): The name of the Foundry project
        location (str): The Azure region where the Foundry account is provisioned
        agent_name (str): The name of the Foundry agent to publish
        agent_guid (str): The GUID of the Foundry agent to publish
        bot_id (str): The Microsoft App ID of the Bot registered in Entra ID for this agent
        app_publish_scope (str): The scope to publish the Teams app to, either "Individual" or "Tenant"
        publish_as_digital_worker (bool): Whether to publish the agent as a Digital Worker in Teams, which surfaces it in the Power Virtual Agents app in addition to allowing it to be installed as a standard Teams app
        app_version (str): The version of the Teams app to publish
        short_description (str): A short description of the agent to display in Teams
        full_description (str): A full description of the agent to display in Teams
        developer_name (str): The name of the developer or organization that created the agent, to display in Teams
        developer_website_url (str): The URL for the developer's website, to display in Teams
        privacy_url (str): The URL for the privacy policy for this agent, to display in Teams
        terms_of_use_url (str): The URL for the terms of use for this agent, to display in Teams
        token (str): The Entra ID access token with the scope of https://ai.azure.com/.default to authenticate the API request
    Returns:
        dict: The response from the Foundry API if the publish was successful, otherwise None
    """

    body = {
        "subscriptionId": subscription_id,
        "agentGuid": agent_guid,
        "agentName": agent_name,
        "appRegistrationId": appRegistrationId,
        "botId": bot_id,
        "appPublishScope": app_publish_scope,
        "publishAsDigitalWorker": publish_as_digital_worker,
        "appVersion": app_version,
        "shortDescription": short_description,
        "fullDescription": full_description,
        "developerName": developer_name,
        "developerWebsiteUrl": developer_website_url,
        "privacyUrl": privacy_url,
        "termsOfUseUrl": terms_of_use_url
    }

    response = requests.post(
        url = f"https://{location}.api.azureml.ms/agent-asset/v2.0/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.MachineLearningServices/workspaces/{account_name}@{project_name}@AML/microsoft365/publish",
        headers={
            "Content-Type": "application/json", 
            "Accept": "application/json",
            "Authorization": f"Bearer {token}",
        },
        json=body
    ) 

    if response.status_code == 200:
        print("Agent published successfully! Status code: 200")
    else:
        logging.error(f"Failed to publish agent: {response.status_code} - {response.text}")
        return None

publish_response = publish_agent_teams(
    subscription_id = os.getenv("FOUNDRY_SUBSCRIPTION_ID"),
    resource_group = os.getenv("FOUNDRY_RESOURCE_GROUP"),
    account_name = os.getenv("FOUNDRY_ACCOUNT_NAME"),
    project_name = os.getenv("FOUNDRY_PROJECT_NAME"),
    location = os.getenv("FOUNDRY_LOCATION"),
    agent_name = os.getenv("FOUNDRY_AGENT_NAME"),
    agent_guid = enabled_agent_guid,
    bot_id = enabled_agent_guid,
    app_publish_scope = "Tenant",
    publish_as_digital_worker = False,
    app_version = "1.0.0",
    short_description = "This is a sample agent published from Foundry to Teams",
    full_description = "This agent was created in Foundry and published to Microsoft Teams using the Foundry API.",
    developer_name = "Carl Carlson",
    developer_website_url = "https://www.example.com",
    privacy_url = "https://www.example.com/privacy",
    terms_of_use_url = "https://www.example.com/terms",
    token = user_token.token
)


# APPENDIX

## Get the list of a user's installed Teams Apps

Teams custom apps can be installed for an individual user. These apps can be [sourced from the Teams app catalog](https://learn.microsoft.com/en-us/microsoftteams/teams-custom-app-policies-and-settings) or [side-loaded](https://learn.microsoft.com/en-us/microsoftteams/platform/concepts/deploy-and-publish/apps-upload).

Foundry Agents installed as Teams Apps for individuals will have a distributionMethod set to **store** while agents installed as Teams Apps for the tenant will have a distributionMethod of **organization**.



In [ ]:
from azure.identity import ClientSecretCredential
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Get a token for Foundry scope
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_ID_TENANT_ID"),
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("CLIENT_SECRET")
)
scopes = ["https://graph.microsoft.com/.default"]

app_graph_token = credential.get_token(*scopes)

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Function that gets the list of Teams apps installed for the user
def get_user_teams_apps(user_object_id: str, token: str):
    """This function retrieves the list of Teams apps installed for a user
    Args:
        user_object_id (str): The object ID of the user in Azure AD
        token (str): The authentication token to use for the API request
    Returns:
        dict: The list of Teams apps installed for the user if the request was successful, otherwise None
    """

    response = requests.get(
        f"https://graph.microsoft.com/v1.0/users/{user_object_id}/teamwork/installedApps?$expand=teamsApp,teamsAppDefinition",

        headers={
            "Content-Type": "application/json", 
            "Authorization": f"Bearer {token}"
        }
    ) 

    if response.status_code == 200:
        return response.json()
    else:
        logging.error(f"Failed to get user installed apps: {response.status_code} - {response.text}")
        return None

user_object_id = "YOUR_USER_OBJECT_ID"
teams_apps = get_user_teams_apps(user_object_id, app_graph_token.token)
print(f"Teams apps installed for user {user_object_id}: {json.dumps(teams_apps, indent=2)}")




## Get the list of apps in the Teams App Catalog

When a user publishes a Foundry Agent to Teams for the entire organization it is not directly published to the Teams app catalog. It must first be approved in the Microsoft 365 Admin Portal. Once approved by an administrator it will be published to the Teams app catalog.

In [ ]:
from azure.identity import ClientSecretCredential
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Get a token for Foundry scope
credential = ClientSecretCredential(
    tenant_id=os.getenv("ENTRA_ID_TENANT_ID"),
    client_id=os.getenv("CLIENT_ID"),
    client_secret=os.getenv("CLIENT_SECRET")
)
scopes = ["https://graph.microsoft.com/.default"]

app_graph_token = credential.get_token(*scopes)

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environmental variables
load_dotenv(override=True)

# Function that get the listing of apps in a tenant Teams app catalog
def get_apps_from_catalog(token: str):
    """This function retrieves the list of Teams apps available in the Teams app catalog for a tenant
    Args:
        token (str): The authentication token to use for the API request
    Returns:
        dict: The list of Teams apps available in the tenant's Teams app catalog if the request was successful, otherwise None
    """

    response = requests.get(
        f"https://graph.microsoft.com/v1.0/appCatalogs/teamsApps?$filter=distributionMethod eq 'organization'",
        headers={
            "Content-Type": "application/json", 
            "Authorization": f"Bearer {token}"
        }
    ) 

    if response.status_code == 200:
        return response.json()
    else:
        logging.error(f"Failed to get user installed apps: {response.status_code} - {response.text}")
        return None

user_object_id = "2e69d9f2-b5b3-482b-9c15-faeca086b632"
teams_apps = get_user_teams_apps(user_object_id, app_graph_token.token)
print(f"Teams apps installed for user {user_object_id}: {json.dumps(teams_apps, indent=2)}")


